In [34]:
try:
    import elasticsearch
    from elasticsearch import Elasticsearch
    
    import pandas as pd
    import json
    from ast import literal_eval
    from tqdm import tqdm
    import datetime
    import os
    import sys
    
    import os
    import tensorflow as tf
    import tensorflow_hub as hub
    import numpy as np
    
    from elasticsearch import helpers
    print("Loaded  .. . . . . . . .")
except Exception as E:
    print("Some Modules are Missing {} ".format(E))

Loaded  .. . . . . . . .


In [35]:
try:
 print("Loaded  s .. . . . . . . .")
 module_url = "/home/forsan/Downloads/universal-sentence-encoder_4" 
 model = hub.load(module_url)    
 print ("module %s loaded" % module_url)
except Exception as E:
    print("Some Modules are Missing {} ".format(E))

Loaded  s .. . . . . . . .
module /home/forsan/Downloads/universal-sentence-encoder_4 loaded


In [36]:
df = pd.read_csv("/home/forsan/Downloads/images (5).csv")


In [37]:
df.head(1)

,id,title
0,2361,A Saudi Arabian Gulf businessman sitting at a ...


In [38]:
def apply_transform(x):
    
    tem = str(x)
    x = tf.constant([tem])
    embeddings = model(x)
    x = np.asarray(embeddings)
    x = x[0].tolist()
    return x

In [42]:
try:
 print('start embdding ...')
 df["title_vector"] = df["title"].apply(apply_transform)
 print('end embdding ...')
except Exception as E:
    print("Some Modules are Missing {} ".format(E))


start embdding ...
end embdding ...


In [43]:
len(df["title_vector"].iloc[0])


512

In [44]:
df.head(1)

,id,title,ml_vector,title_vector
0,2361,A Saudi Arabian Gulf businessman sitting at a ...,"[-0.03021049126982689, 0.06228601932525635, -0...","[-0.03021049126982689, 0.06228601932525635, -0..."


In [45]:
Settings = {

   "mappings":{
       "properties":{
            "title": {
              "type": "text"
            },
            "title_vector": {
              "type": "dense_vector",
              "dims": 512
            }
    }
   }
}

In [46]:
ENDPOINT = "http://localhost:9200/"
es = Elasticsearch(timeout=600,hosts=ENDPOINT)

In [47]:
es.ping()

True

In [48]:
IndexName = 'arabstockimgdeven'
my = es.indices.create(index=IndexName, ignore=[400,404], body=Settings)

In [49]:
my

{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'arabstockimgdeven'}

In [50]:
df.columns


Index(['id', 'title', 'ml_vector', 'title_vector'], dtype='object')

In [53]:
def generator(df2):
    for c, line in enumerate(df2):
        yield {
    '_index': 'arabstockimgdeven',
   # '_type': '_doc',
    '_id': c+1,
    '_source': {
        "title":line.get("title", ""),
       'id':line.get('id', ""),
        'title_vector':line.get('title_vector', "")
    }
        }
    return 1

In [54]:
df22 = df.to_dict('records')
next(generator(df22))

{'_index': 'arabstockimgdeven',
 '_id': 1,
 '_source': {'title': 'A Saudi Arabian Gulf businessman sitting at a desk in front of a laptop, talking through a mobile phone, a Saudi company, an office job, a work environment',
  'id': 2361,
  'title_vector': [-0.03021049126982689,
   0.06228601932525635,
   -0.03524915128946304,
   0.02693304978311062,
   0.05703500658273697,
   -0.06953904777765274,
   0.042686671018600464,
   0.023693442344665527,
   0.06772022694349289,
   -0.04854009300470352,
   -0.011909657157957554,
   -0.014479989185929298,
   -0.019064899533987045,
   -0.014340095221996307,
   0.023884151130914688,
   -0.07175423204898834,
   -0.007888462394475937,
   0.03132487088441849,
   0.05627507343888283,
   -0.0004308786301407963,
   -0.0048030223697423935,
   -0.008519348688423634,
   0.06513410061597824,
   -0.04762867093086243,
   0.022342585027217865,
   -0.03370828181505203,
   -0.06193926930427551,
   0.0030579061713069677,
   0.005796200595796108,
   -0.06411945074

In [55]:
try:
    print("start working")
    res = helpers.bulk(es, generator(df22))
    print("end Working")
except Exception as E:
    print("Some Modules are Missing {} ".format(E))

start working
end Working


In [60]:
title = "Krish Trish and Baltiboy: Best Friends Forever"
INPUT = input("Enter input query")

x = apply_transform(INPUT)
Query = {
   "_source":[
      "title","id"
   ],
   "size":3000,
   "query":{
      "script_score":{
         "query":{
           # "match":{"title":INPUT}
             "match_all": {}
         },
         "script":{
            "source":"cosineSimilarity(params.query_vector, 'title_vector') + 1.0",
       #  "source": """
        #  double value = dotProduct(params.query_vector, 'ml_vector');
         # return sigmoid(1, Math.E, -value); 
        #""",
                     #"source": "1 / (1 + l1norm(params.query_vector, 'ml_vector'))", 

            "params":{
               "query_vector":x

            }
         }
      }
   }
}


resp = es.search(index="arabstockimgdeven",  body=Query)

print("Got %d Hits:" % resp['hits']['total']['value'])
#print(resp)
for hit in resp['hits']['hits']:
    print(  hit["_source"])

Enter input query horse


Got 10000 Hits:
{'id': 35591, 'title': 'Close-up of a black thoroughbred Arabian horse, horse breeding and training, thoroughbred Arabian horse, horse stable, horse farm'}
{'id': 17694, 'title': 'Close-up of a purebred Arabian horse, dressage, horse training, purebred Arabian horse, horse riding, equestrian'}
{'id': 17734, 'title': 'Close-up of a purebred Arabian horse, dressage, horse training, purebred Arabian horse, horse riding, equestrian'}
{'id': 35595, 'title': 'Close-up of a purebred white Arabian horse, horse breeding and training, thoroughbred Arabian horse, horse stable, horse farm'}
{'id': 44661, 'title': 'Close-up of a purebred Arabian horse, a beautiful Arabian horse, breeding and training horses, thoroughbred Arabian horses, horse stables, horse farm'}
{'id': 44662, 'title': 'Close-up of a purebred Arabian horse, a beautiful Arabian horse, breeding and training horses, thoroughbred Arabian horses, horse stables, horse farm'}
{'id': 35592, 'title': 'Close-up of the face o